# 08 — P&L Attribution

I separate exact ledger reconciliation from the Greek and variance approximations. That distinction makes it clear which statements are accounting identities and which are economic explanations.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))


In [2]:
from options_lab.price_paths import simulate_gbm_paths
from options_lab.delta_hedging import delta_hedge_path
from options_lab.pnl_attribution import attribute_cash_ledger, counterfactual_pnl_attribution, greek_path_attribution, gamma_variance_attribution

times,paths=simulate_gbm_paths(100.,.03,.20,1.,252,1,seed=2026)
result=delta_hedge_path(times,paths[0],100.,.03,.20,"call","short",transaction_cost_rate=.0005)
ledger=attribute_cash_ledger(result)
pd.Series(ledger.__dict__)

option_entry_cashflow         9.413403e+00
stock_trading_cashflow        2.125215e+01
option_settlement_cashflow   -2.885157e+01
financing_cashflow           -2.358541e+00
transaction_cost_pnl         -2.971076e-01
reconstructed_terminal_pnl   -8.416647e-01
reported_terminal_pnl        -8.416647e-01
reconciliation_error         -6.483702e-14
dtype: float64

In [3]:
counter=counterfactual_pnl_attribution(times,paths[0],100.,.03,.20,"call","short",
                                         transaction_cost_rate=.0005,
                                         actual_option_premium=result.option_premium_per_unit+.25)
pd.Series(counter.__dict__)

model_premium_per_unit             9.413403
actual_premium_per_unit            9.663403
model_hedging_pnl                 -0.539958
premium_edge_at_inception          0.250000
premium_edge_at_expiry             0.257614
transaction_cost_drag_at_expiry   -0.301707
nominal_transaction_cost           0.297108
transaction_cost_future_value      0.301707
attributed_terminal_pnl           -0.584051
actual_terminal_pnl               -0.584051
reconstruction_error               0.000000
dtype: float64

In [4]:
greek=greek_path_attribution(result,100.,.03,.20,"call","short")
model_no_cost=delta_hedge_path(times,paths[0],100.,.03,.20,"call","short")
variance=gamma_variance_attribution(model_no_cost,100.,.03,.20,"call","short")
pd.Series({"greek_taylor_residual":greek.taylor_residual,
           "greek_reconciliation_error":greek.reconciliation_error,
           "gamma_variance_pnl":variance.total_variance_pnl,
           "exact_model_hedging_pnl":variance.model_hedging_pnl,
           "gamma_variance_residual":variance.residual_pnl})

greek_taylor_residual         3.226334e-02
greek_reconciliation_error   -6.772360e-14
gamma_variance_pnl           -5.440533e-01
exact_model_hedging_pnl      -5.399579e-01
gamma_variance_residual       4.095384e-03
dtype: float64

**What I took from it.** The cash ledger and counterfactual identities reconcile exactly. Greek and variance attribution are approximations, and their residuals contain higher-order, discrete-time, and path effects.